In [ ]:
%load_ext autoreload
%autoreload 2

import os
import json
import numpy as np
import polars as pl
import pandas as pd
import pickle as pkl
from tqdm import tqdm

from plotnine import *
import matplotlib.pyplot as plt

## PRSs

In [ ]:
qt_ref_df = pl.read_csv('PATH_TO_FILE', separator=' ', has_header=False)
qt_ref_df

In [ ]:
def file2df(path: str) -> pl.DataFrame:
    # Read the file once
    with open(path, "r") as f:
        lines = [line.strip().split() for line in f.readlines()]

    # Row 0 becomes header
    header = [lines[0][0], lines[1][0]]
    # Pair up the rest
    col1 = lines[0][1:]
    col2 = lines[1][1:]

    return pl.DataFrame({header[0]: col1, header[1]: col2})

path = 'PATH_TO_FILE'
file2df(path).rename({'0': 'Mothers_age_at_death'})

In [ ]:
regenie_dir = 'PATH_TO_FILE'
ref_df = pl.read_csv(f'{regenie_dir}prs.list_prs.list', separator=' ', has_header=False)

qt_list = []
for row in ref_df.iter_rows():
    trait = row[0]
    trait_file = row[1]
    t = file2df(trait_file).rename({'0': 'prs'}).with_columns(
        pl.col('prs').cast(pl.Float32),
        pl.lit(f'{trait.lower()}_int_prs').alias('trait')
    )

    qt_list.append(t)

qt_df = pl.concat(qt_list, how='vertical').with_columns(
    pl.col('FID_IID').str.split('_').list.first().alias('FID'),
    pl.col('FID_IID').str.split('_').list.last().alias('IID'),
).with_columns(
    pl.col('FID').cast(pl.Int32),
    pl.col('IID').cast(pl.Int32)
).drop('FID_IID')

qt_df

In [ ]:
regenie_dir = 'PATH_TO_FILE'
ref_df = pl.read_csv(f'{regenie_dir}prs.list_prs.list', separator=' ', has_header=False)

bt_list = []
for row in ref_df.iter_rows():
    trait = row[0]
    trait_file = row[1]
    t = file2df(trait_file).rename({'0': 'prs'}).with_columns(
        pl.lit(f'{trait.lower()}_int_prs').alias('trait'),
        pl.when(pl.col("prs") == "NA")
        .then(pl.lit(None))
        .otherwise(pl.col("prs"))
        .cast(pl.Float32)
        .alias("prs")
    )

    bt_list.append(t)

bt_df = pl.concat(bt_list, how='vertical').with_columns(
    pl.col('FID_IID').str.split('_').list.first().alias('FID'),
    pl.col('FID_IID').str.split('_').list.last().alias('IID'),
).with_columns(
    pl.col('FID').cast(pl.Int32),
    pl.col('IID').cast(pl.Int32)
).drop('FID_IID')

bt_df

In [ ]:
prs_df = pl.concat([qt_df, bt_df], how='vertical')
prs_df

In [ ]:
prs_df_wide = prs_df.pivot(
    index=['FID','IID'],
    on='trait',
    values='prs'
)

prs_df_wide

In [ ]:
# prs_df_wide.write_parquet('PATH_TO_FILE')

## RVAT

In [ ]:
regenie_dir = 'PATH_TO_FILE'
qt_list = []
for trait in os.listdir(regenie_dir):
    trait_file = os.path.join(regenie_dir, trait)
    if trait_file.endswith('.regenie'):
        t = pl.read_csv(trait_file, separator=' ').with_columns(
                pl.lit(trait.split('__')[1].split('.')[0].lower()).alias('trait')
            )
        qt_list.append(t)

qt_df = pl.concat(qt_list, how='vertical').rename(
            {col: col.lower() for col in qt_list[0].columns}
        ).with_columns(
            pl.lit('quantitative').alias('trait_type')
        )
         
qt_df

In [ ]:
regenie_dir = 'PATH_TO_FILE'
bt_list = []
for trait in os.listdir(regenie_dir):
    trait_file = os.path.join(regenie_dir, trait)
    if trait_file.endswith('.regenie'):
        t = pl.read_csv(trait_file, separator=' ').with_columns(
                pl.lit(trait.split('__')[1].split('.')[0].lower()).alias('trait')
            )
        bt_list.append(t)

bt_df = pl.concat(bt_list, how='vertical').rename(
            {col: col.lower() for col in bt_list[0].columns}
        ).with_columns(
            pl.lit('binary').alias('trait_type')
        )
bt_df

In [ ]:
gnames = pl.read_parquet('PATH_TO_FILE').filter(
    (pl.col('uniprotkb_gene_name_id').is_not_null()) &
    (pl.col('ensembl_canonical').is_not_null())
).select(
    ['gene_stable_id', 'gene_name']
).unique().rename({"gene_stable_id": "gene_id"})

gnames

In [ ]:
rvat_df = pl.concat([qt_df, bt_df], how='vertical').rename(
    {'id': 'gene_id', 'log10p': 'neg_log10p', 'a1freq': 'plof_freq', 'n': 'n_samples'}
).with_columns(
    (pl.col('neg_log10p') > -np.log10(0.05)).alias('nom_significance')
).drop(['genpos', 'allele0', 'allele1', 'info', 'extra', 'test'])

rvat_df = gnames.join(rvat_df, on='gene_id', how='right')

rvat_df = rvat_df.lazy().join(
    rvat_df.lazy()
    .group_by("trait")
    .agg(pl.count().alias("n_genes")),
    on="trait",
    how="left"
).with_columns(
    (pl.col('neg_log10p') > -np.log10(0.05/pl.col('n_genes'))).alias('bonf_significance')
).collect()

rvat_df

In [ ]:
# rvat_df.write_parquet('PATH_TO_FILE')

In [ ]:
# 1. Extract as numpy
# obs_neglog10 = rvat_df.filter(pl.col('nom_significance')==True).filter(pl.col('trait_type')=='quantitative')["neg_log10p"].to_numpy()
obs_neglog10 = rvat_df.filter(pl.col('nom_significance')==True).filter(pl.col('trait_type')=='binary')["neg_log10p"].to_numpy()

# 2. Sort observed values
obs_sorted = np.sort(obs_neglog10)

# 3. Expected values under uniform(0,1)
n = len(obs_sorted)
expected = -np.log10((np.arange(1, n + 1)) / (n + 1))

# 4. Convert to pandas for plotnine
qq_df = pd.DataFrame({
    "expected": expected[::-1],
    "observed": obs_sorted
})

# 5. QQ plot
plot = (
    ggplot(qq_df, aes(x="expected", y="observed"))
    + geom_point(size=1.2, alpha=0.6)
    + geom_abline(slope=1, intercept=0, color="red", linetype="dashed")
    + labs(
        x="Expected -log10(p)",
        y="Observed -log10(p)",
        title="QQ Plot of p-values"
    )
    + theme_538()
)

plot